# Human lymph node — CellSTIC tutorial
This notebook organizes the **human lymph node (RNA + ADT)** workflow into a clean, reproducible notebook structure.

## 1. Repository path

This cell resolves the project root so notebook execution works both inside and outside script-style contexts.


In [ ]:
from pathlib import Path
import sys
import torch

# Add project root to path
try:
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project_root = {project_root}")
print(f"python executable available, torch version = {torch.__version__}")


## 2. Imports

All project-level imports are kept in one place so dependency issues surface early.


In [ ]:
from utils.tools.seed_utils import set_global_seed
from utils.loader import load_human_lymph_node
from utils.train import load_config
from model.cellstic import CellSTIC
from pipeline.trainer import CellSTICTrainer
from pipeline.evaluator import CellSTICEvaluator
from utils import ModelUtils
from utils.tools import CellChatDBLoader
from pipeline.analyzer import TreeLevelAnalysis
from pipeline.analyzer import SingleLevelAnalysis
from utils.metrics import SpatialVisualizer, MetricsComputer, UMAPVisualizer, get_custom_palette

set_global_seed()  # once before train/eval; pipeline does not set RNG here


## 3. Runtime parameters

Edit this section first when adapting the notebook to a new machine or rerun mode.


In [ ]:
WORK_DIR = str(project_root / "data" / "human_lymph_node")
RUN_TRAIN = True
USE_CACHE = True
RUN_MODALITY_UMAP = True
RUN_CELL_TYPE_VIS = True
TREE_THRESHOLD = 0.8

work_dir = Path(WORK_DIR)
output_path = work_dir / "output"
raw_path = work_dir / "raw"
model_path = work_dir / "model"
preprocess_path = work_dir / "preprocess"
config_path = work_dir / "config" / "human_lymph_node_config.yaml"

for _p in [output_path, raw_path, model_path, preprocess_path]:
    _p.mkdir(parents=True, exist_ok=True)

print(f"WORK_DIR      = {work_dir}")
print(f"config_path   = {config_path}")
print(f"output_path   = {output_path}")
print(f"RUN_TRAIN     = {RUN_TRAIN}")
print(f"USE_CACHE     = {USE_CACHE}")
print(f"TREE_THRESHOLD= {TREE_THRESHOLD}")

## 4. Load config, resources, and dataset

This is the main data-loading cell used by the training and analysis pipeline.


In [ ]:
if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

config = load_config(config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cell_chat_db = CellChatDBLoader().get_ligand_receptor_map("human")
rna_adata, adt_adata, ligand_receptor_map = load_human_lymph_node(
    raw_path,
    preprocess_path,
    use_cache=USE_CACHE,
)

print(f"device = {device}")
print(f"rna_adata shape = {rna_adata.shape}")
print(f"adt_adata shape = {adt_adata.shape}")
print(f"rna obs columns (sample) = {list(rna_adata.obs.columns[:10])}")
print(f"adt obs columns (sample) = {list(adt_adata.obs.columns[:10])}")
print(f"ligand_receptor_map entries = {len(ligand_receptor_map)}")


## 5. Optional modality UMAP export

This section reproduces the per-modality UMAP export from the original runner workflow.


In [ ]:
if RUN_MODALITY_UMAP:
    rna_adata_umap, adt_adata_umap, _ = load_human_lymph_node(
        raw_path,
        preprocess_path,
        use_cache=USE_CACHE,
    )

    modality_map = {
        "RNA": rna_adata_umap,
        "ADT": adt_adata_umap,
    }

    for modality_name, adata in modality_map.items():
        if "feat" not in adata.obsm:
            raise ValueError(f"{modality_name} missing adata.obsm['feat'], cannot run per-modality UMAP.")
        MetricsComputer.run_region_umap_metrics_export(
            adata=adata,
            save_dir=output_path / modality_name.lower(),
            feature_key="feat",
            true_labels=None,
            cluster_key=None,
            n_clusters=6,
        )
        print(f"Saved modality UMAP outputs for {modality_name} to {output_path / modality_name.lower()}")
else:
    print("Skipped modality UMAP export.")


## 6. Optional cell-type spatial distribution and UMAP

This section reproduces the visualization utilities from the original runner script without changing the underlying internal calls.


In [ ]:
if RUN_CELL_TYPE_VIS:
    rna_adata_ct, _, _ = load_human_lymph_node(raw_path, preprocess_path, use_cache=USE_CACHE)
    if "cell_type" not in rna_adata_ct.obs:
        raise ValueError("`cell_type` not found in rna_adata.obs; cannot visualize cell-type spatial distribution.")

    rna_adata_ct = rna_adata_ct.copy()
    rna_adata_ct.obs["cluster"] = rna_adata_ct.obs["cell_type"].astype(str).astype("category")

    save_path_ct = output_path / "cell_type_distribution" / "spatial_domain.svg"
    save_path_ct.parent.mkdir(parents=True, exist_ok=True)
    palette = get_custom_palette(len(rna_adata_ct.obs["cluster"].cat.categories))

    SpatialVisualizer.generate_spatial_domain_visualization(
        adata_source=rna_adata_ct,
        save_path=str(save_path_ct),
        palette=palette,
    )
    print(f"Cell-type spatial distribution saved to {save_path_ct}")

    if "feat" not in rna_adata_ct.obsm:
        raise ValueError("`feat` not found in rna_adata.obsm; cannot generate UMAP for cell types.")

    umap_path_ct = output_path / "cell_type_distribution" / "umap.svg"
    UMAPVisualizer.generate_visualization(
        node_features=torch.tensor(rna_adata_ct.obsm["feat"], dtype=torch.float32),
        adata_source=rna_adata_ct,
        save_path=str(umap_path_ct),
        palette=palette,
        add_kde_contour=False,
        add_cluster_labels=False,
        n_neighbors=40,
        color_key="cluster",
    )
    print(f"Cell-type UMAP saved to {umap_path_ct}")
else:
    print("Skipped cell-type visualization.")


## 7. Train model or load checkpoint

Set `RUN_TRAIN = False` to reuse an existing saved model.


In [ ]:
if RUN_TRAIN:
    model = CellSTIC(config.model, device)
    CellSTICTrainer(
        model,
        config,
        model_path=model_path,
        ligand_receptor_map=ligand_receptor_map,
        device=device,
        cell_chat_db=cell_chat_db,
    ).train(
        modality_datas=[rna_adata, adt_adata],
        is_train_ccc=True,
        is_train_feature=True,
    )
    print(f"Training finished. Model artifacts saved under {model_path}")
else:
    model = ModelUtils.load_model(model_path=model_path, config=config, device=device)
    print(f"Loaded existing model from {model_path}")


## 8. Evaluate tree-level CCC

This cell computes the tree-level CCC outputs used by the downstream analysis modules.


In [ ]:
evaluator = CellSTICEvaluator(
    model,
    config,
    ligand_receptor_map=ligand_receptor_map,
    model_path=model_path,
    output_path=output_path,
    device=device,
)

results = evaluator.evaluate_ccc_precision_tree(
    modality_datas=[rna_adata, adt_adata],
)

print(f"Number of tree levels returned = {len(results)}")
for idx, level_result in enumerate(results, start=1):
    print(f"tree_level_{idx}: keys = {list(level_result.keys())}")


## 9. Tree-level alluvial analysis

This section generates tree-level summaries such as alluvial plots.


In [ ]:
# Tree-level analyzer over multi-level CCC `results` (same output_path / threshold as above)
tree_analyzer = TreeLevelAnalysis(
    tree_level_results=results,
    output_path=output_path,
    threshold=TREE_THRESHOLD,
    cell_type_key="cell_type",
)

# Alluvial diagrams: how LR signals flow between sender–receiver cell-type pairs across tree levels
tree_analyzer.run_alluvial_per_cell_type_pair()

print(f"Tree-level analysis completed with threshold = {TREE_THRESHOLD}")

## 10. Single-level heatmaps

This section exports cell-type heatmaps for each tree level.


In [ ]:
# Shared SingleLevelAnalysis instance; per-call methods supply level-specific AnnData and edge tensors
analyzer = SingleLevelAnalysis(
    adata=None,
    pos_edge_probs_np=None,
    edge_type_map=None,
    output_path=output_path,
    cell_type_key="cell_type",
    threshold=TREE_THRESHOLD,
)

# Tree level 1: simplified LR heatmaps (root-level LR filter)
analyzer.run_simple_heatmaps(
    adata=results[0]["adata"],
    pos_edge_probs_np=results[0]["pos_edge_probs_np"],
    edge_type_map=results[0]["edge_type_map"],
    output_path=output_path / "tree_level_1",
    lr_filter=["root"],
    font_size=18,
)

# Tree level 2: simplified LR heatmaps (all LR pairs)
analyzer.run_simple_heatmaps(
    adata=results[1]["adata"],
    pos_edge_probs_np=results[1]["pos_edge_probs_np"],
    edge_type_map=results[1]["edge_type_map"],
    output_path=output_path / "tree_level_2",
    lr_filter=None,
    font_size=18,
)

print("Single-level heatmap export completed.")
